In [ ]:
import torch
import cv2
import numpy as np
import math
import matplotlib.pyplot as plt
import glob
import os
import pandas as pd
import segmentation_models_pytorch as smp 
import scipy.stats as stats
import time
begin = time.time()
# ================= CONFIGURATION =================
MODEL_PATH = "titanium_unet_weights_update.pth" 
IMAGE_FOLDER = "YOUR IMAGE FOLDER" ##<-------------- UPDATE 
OUTPUT_CSV_NAME = "titanium_analysis_results.csv"

MICRONS_PER_PIXEL = 0.02336448 
SCALE_BAR_LENGTH_MICRONS = 1 

PATCH_SIZE = 256
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ================= HELPER FUNCTIONS =================

def opencv_skeletonize(img):
    img = img.copy()
    skel = np.zeros(img.shape, np.uint8)
    element = cv2.getStructuringElement(cv2.MORPH_CROSS, (3,3))
    while True:
        eroded = cv2.erode(img, element)
        temp = cv2.dilate(eroded, element)
        temp = cv2.subtract(img, temp)
        skel = cv2.bitwise_or(skel, temp)
        img = eroded.copy()
        if cv2.countNonZero(img) == 0:
            break
    return skel

def measure_intercept_inverse_stereology(mask, num_lines=100):
    """
    Calculates True 3D Thickness using the Mean Inverse Intercept method.
    Formula: t = (2/3) * [1 / Mean(1/L_intercept)]
    """
    h, w = mask.shape
    all_intercept_lengths = []
    total_length_pixels = 0
    total_intercepts_count = 0   #for measuring alpha lath width
    total_intercepts_trans = 0   #for emasure all phase boundary transitions
    
    for _ in range(num_lines):
        x1, y1 = np.random.randint(0, w), np.random.randint(0, h)
        x2, y2 = np.random.randint(0, w), np.random.randint(0, h)
        
        length_px = int(np.hypot(x2 - x1, y2 - y1))
        if length_px < 10: continue 
        
        x_values = np.linspace(x1, x2, length_px).astype(int)
        y_values = np.linspace(y1, y2, length_px).astype(int)
        
        profile = mask[np.clip(y_values, 0, h-1), np.clip(x_values, 0, w-1)]
        binary_profile = (profile == 255).astype(np.uint8)
        transitions = np.sum(np.abs(np.diff(profile)) > 0)
        padded = np.pad(binary_profile, (1, 1), 'constant')
        diff = np.diff(padded.astype(int))
        
        starts = np.where(diff == 1)[0]
        ends = np.where(diff == -1)[0]
        
        lengths = ends - starts
        valid_lengths = lengths[lengths > 1]
        all_intercept_lengths.extend(valid_lengths)
        
        total_length_pixels += length_px
        total_intercepts_count += len(valid_lengths)
        total_intercepts_trans += transitions

    if not all_intercept_lengths:
        return 0, 0, total_length_pixels, 0

    inv_lengths = 1.0 / np.array(all_intercept_lengths)
    mean_inv_intercept = np.mean(inv_lengths)
    true_3d_thickness_px = (2.0/3.0) * (1.0 / mean_inv_intercept)
    mli_px = total_length_pixels/total_intercepts_trans

    return true_3d_thickness_px, mli_px, total_length_pixels, total_intercepts_count

def add_scale_bar(image, um_per_px, bar_length_um):
    h, w = image.shape[:2]
    bar_pixels = int(bar_length_um / um_per_px)
    margin = 40
    start_point = (w - margin - bar_pixels, h - margin)
    end_point = (w - margin, h - margin)
    
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    
    cv2.line(image, start_point, end_point, (0, 255, 0), 6)
    cv2.putText(image, f"{bar_length_um} um", (start_point[0], start_point[1] - 15), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
    return image

def process_large_image(image_path, model, patch_size=256, stride=128):
    original_img = cv2.imread(image_path)
    if original_img is None: return None, None, None
    img_rgb = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    
    pad_h = (math.ceil(h / patch_size) * patch_size) - h
    pad_w = (math.ceil(w / patch_size) * patch_size) - w
    img_padded = cv2.copyMakeBorder(img_rgb, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)
    ph, pw = img_padded.shape[:2]
    
    full_mask_acc = np.zeros((ph, pw), dtype=np.float32)
    full_var_acc = np.zeros((ph, pw), dtype=np.float32)
    count_map = np.zeros((ph, pw), dtype=np.float32)
    
    with torch.no_grad():
        for y in range(0, ph - patch_size + 1, stride):
            for x in range(0, pw - patch_size + 1, stride):
                patch = img_padded[y:y+patch_size, x:x+patch_size]
                input_tensor = patch.transpose(2, 0, 1).astype('float32') / 255.0
                input_tensor = torch.from_numpy(input_tensor).unsqueeze(0).to(DEVICE)
                
                p1 = torch.sigmoid(model(input_tensor)).squeeze().cpu().numpy()
                logits_h = model(torch.flip(input_tensor, dims=[3]))
                p2 = torch.flip(torch.sigmoid(logits_h), dims=[3]).squeeze().cpu().numpy()
                logits_v = model(torch.flip(input_tensor, dims=[2]))
                p3 = torch.flip(torch.sigmoid(logits_v), dims=[2]).squeeze().cpu().numpy()
                logits_hv = model(torch.flip(input_tensor, dims=[2, 3]))
                p4 = torch.flip(torch.sigmoid(logits_hv), dims=[2, 3]).squeeze().cpu().numpy()
                
                stacked_probs = np.stack([p1, p2, p3, p4], axis=0)
                full_mask_acc[y:y+patch_size, x:x+patch_size] += np.mean(stacked_probs, axis=0)
                full_var_acc[y:y+patch_size, x:x+patch_size] += np.var(stacked_probs, axis=0)
                count_map[y:y+patch_size, x:x+patch_size] += 1.0

    final_mask_prob = full_mask_acc / np.maximum(count_map, 1)
    final_mask_binary = (final_mask_prob[:h, :w] > 0.5).astype(np.uint8) * 255
    final_var = (full_var_acc / np.maximum(count_map, 1))[:h, :w]
    overall_confidence = (1.0 - (np.mean(final_var) / 0.25)) * 100
    
    return original_img, final_mask_binary, overall_confidence

# ================= MAIN EXECUTION =================

def main():
    print(f"Loading model on {DEVICE}...")
    model = smp.Unet(encoder_name="resnet18", encoder_weights=None, in_channels=3, classes=1, activation=None)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
    model.to(DEVICE).eval()
    
    image_files = []
    for ext in ["*.tif", "*.jpg", "*.png", "*.bmp"]:
        image_files.extend(glob.glob(os.path.join(IMAGE_FOLDER, ext)))
    image_files = [f for f in image_files if "_processed" not in f]
    
    print(f"Found {len(image_files)} images to process.")
    results_data = []

    for idx, img_path in enumerate(image_files):
        filename = os.path.basename(img_path)
        print(f"[{idx+1}/{len(image_files)}] Processing {filename}...")
        
        original, mask, overall_confidence = process_large_image(img_path, model)
        if mask is None: continue 
            
        # A. Phase Fraction
        alpha_fraction = (cv2.countNonZero(mask) / (mask.shape[0] * mask.shape[1])) * 100
        
        # B. 2D Skeleton Width
        dist_map = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
        skeleton = opencv_skeletonize(mask)
        widths_px = dist_map[skeleton > 0] * 2
        avg_width_2d = np.mean(widths_px[widths_px > 2.0]) * MICRONS_PER_PIXEL if len(widths_px[widths_px > 2.0]) > 0 else 0

        # C. 3D Stereology (2/3*InverseMean Linear Intercept)
        thickness_3d_px, mli_px, total_len_px, total_ints = measure_intercept_inverse_stereology(mask, num_lines=100)
        
        true_3d_thickness_um = thickness_3d_px * MICRONS_PER_PIXEL
        mli_um = mli_px * MICRONS_PER_PIXEL
        total_len_um = total_len_px * MICRONS_PER_PIXEL

        # Saving Outputs
        base_name = os.path.splitext(filename)[0]
        mask_with_scale = add_scale_bar(mask.copy(), MICRONS_PER_PIXEL, SCALE_BAR_LENGTH_MICRONS)
        cv2.imwrite(os.path.join(IMAGE_FOLDER, f"{base_name}_processed.png"), mask_with_scale)
        #Save with no scale bar
        #cv2.imwrite(os.path.join(IMAGE_FOLDER, f"{base_name}_processed.png"), mask.copy())
        
        results_data.append({
            "Filename": filename,
            "Overall Confidence (%)": round(overall_confidence, 2),
            "Alpha Phase (%)": round(alpha_fraction, 2),
            "Avg Lath Width (2D Skeleton)": round(avg_width_2d, 4),
            "Adjusted 3D Lath Thickness (Stereology)": round(true_3d_thickness_um, 4),
            "Mean Linear Intercept (um)": round(mli_um, 4),
            "Total Line Length (um)": round(total_len_um, 2),
            "Total Intercepts": int(total_ints)
        })

    if results_data:
        df = pd.DataFrame(results_data)
        df.to_csv(os.path.join(IMAGE_FOLDER, OUTPUT_CSV_NAME), index=False)
        
        n = len(df)
        if n > 1:
            t_val = stats.t.ppf(1 - 0.025, n - 1) 
            metrics = ['Alpha Phase (%)', 'Avg Lath Width (2D Skeleton)', 'Adjusted 3D Lath Thickness (Stereology)','Mean Linear Intercept (um)']
            summary = []
            for m in metrics:
                mean_v, std_v = df[m].mean(), df[m].std(ddof=1)
                ci_95 = (t_val * std_v) / math.sqrt(n)
                ra = (ci_95 / mean_v) * 100 if mean_v > 0 else 0
                summary.append({"Metric": m, "Avg": round(mean_v, 4), "Std Dev": round(std_v, 4), "95% CI": round(ci_95, 4), "Relative Accuracy%": round(ra, 2)})
            
            summary_df = pd.DataFrame(summary)
            summary_df.to_csv(os.path.join(IMAGE_FOLDER, "folder_summary_statistics.csv"), index=False)
            print("\n--- Folder Summary ---\n", summary_df.to_string(index=False))

if __name__ == "__main__":
    main()
time.sleep(1)
end = time.time()
print (f"Total runtime of the program is {end-begin}")